# 🧪 Laboratorio 8: Segmentación de Perfiles Docentes — Soluciones

**Módulo 03** | **Sesión 6** | **Duración estimada: 45min**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-03-modelado-predictivo/laboratorios/lab_08_clustering_soluciones.ipynb)

---

Este notebook contiene las **soluciones completas** con código, interpretaciones y comentarios didácticos.

## 🎯 Objetivos de Aprendizaje

| # | Competencia | Descripción | Nivel |
|---|-------------|-------------|-------|
| 1 | Preparación de datos | Seleccionar, limpiar y estandarizar variables para clustering | Aplicación |
| 2 | Selección de K | Aplicar el método del codo y silhouette para elegir K | Análisis |
| 3 | K-Means | Ejecutar K-Means y analizar los clusters resultantes | Aplicación |
| 4 | Perfilamiento | Construir una tabla de perfiles y nombrar cada cluster | Evaluación |
| 5 | Comunicación | Traducir los hallazgos en recomendaciones de gestión de talento | Síntesis |

## 💡 ¿Por qué es importante para tu práctica docente?

La **segmentación** es una herramienta poderosa para la gestión del talento docente. En lugar de tratar a todos los docentes de FACES igual, este análisis permite identificar **perfiles diferenciados** y diseñar intervenciones específicas:

- Docentes con alta carga y baja satisfacción necesitan alivio de carga
- Docentes jóvenes y productivos necesitan mentoría y oportunidades de crecimiento
- Docentes experimentados y satisfechos pueden servir como mentores

Al completar este laboratorio, habrás construido un análisis que podrías presentar al Decanato de FACES.

In [ ]:
# Librerías necesarias — ejecuta esta celda primero
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Librerías cargadas correctamente ✓')

---

## Ejercicio 1: Seleccionar features, verificar datos y estandarizar

In [ ]:
# ============================================================
# SOLUCIÓN Ejercicio 1: Seleccionar features y estandarizar
# ============================================================

# 1. Cargar el dataset de encuesta docente
df = pd.read_csv('../../datasets/universidad/encuesta_docente.csv')
print(f'📊 Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'\nColumnas: {list(df.columns)}')
display(df.head())

# 2. Seleccionar las columnas numéricas relevantes para clustering
cols_clustering = [
    'satisfaccion_general', 'satisfaccion_salarial',
    'satisfaccion_infraestructura', 'satisfaccion_desarrollo',
    'publicaciones', 'carga_horaria', 'antiguedad'
]
df_cluster = df[cols_clustering].copy()

# 3. Verificar datos faltantes
faltantes = df_cluster.isnull().sum()
print(f'\n❓ Valores faltantes por columna:')
print(faltantes)

# Eliminar filas con NaN si las hay
if faltantes.sum() > 0:
    df_cluster.dropna(inplace=True)
    print(f'\nRegistros después de eliminar NaN: {len(df_cluster)}')
else:
    print('\n✅ No hay valores faltantes')

# 4. Estandarizar con StandardScaler
#    K-Means es sensible a la escala, así que estandarizamos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster)

# 5. Verificar que media ≈ 0 y std ≈ 1
print(f'\n📊 Verificación de estandarización:')
print(f'  Medias:  {X_scaled.mean(axis=0).round(4)}')
print(f'  Std:     {X_scaled.std(axis=0).round(4)}')
print(f'\n✅ Todas las variables tienen media ≈ 0 y desviación estándar ≈ 1')

---

## Ejercicio 2: Método del Codo (Elbow Plot)

In [ ]:
# ============================================================
# SOLUCIÓN Ejercicio 2: Método del Codo
# ============================================================

# 1. Ajustar K-Means para diferentes valores de K
rango_k = range(2, 9)  # K = 2, 3, 4, 5, 6, 7, 8
inercias = []

for k in rango_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inercias.append(km.inertia_)

# 2. Guardar la inercia de cada modelo (ya hecho arriba)

# 3. Gráfico del codo
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(rango_k, inercias, 'bo-', linewidth=2, markersize=8)
ax.set_xlabel('Número de clusters (K)')
ax.set_ylabel('Inercia (suma de distancias intra-cluster)')
ax.set_title('Método del Codo — Selección de K')
ax.set_xticks(list(rango_k))
plt.tight_layout()
plt.show()

# 4-5. Identificar el codo
print('📊 Inercia por K:')
for k, inercia in zip(rango_k, inercias):
    print(f'  K={k}: {inercia:.1f}')

print(f'\n📌 El "codo" de la curva parece estar en K=3 o K=4.')
print(f'   A partir de ese punto, agregar más clusters reduce poco la inercia.')

---

## Ejercicio 3: Análisis de Silhouette

In [ ]:
# ============================================================
# SOLUCIÓN Ejercicio 3: Análisis de Silhouette
# ============================================================

# 1. Calcular silhouette score para cada K
silhouettes = []

for k in rango_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    silhouettes.append(sil)

# 2. Tabla de resultados
print('📊 Silhouette Score por K:')
tabla_sil = pd.DataFrame({
    'K': list(rango_k),
    'Silhouette Score': [round(s, 4) for s in silhouettes]
})
display(tabla_sil)

# 3. Gráfico de K vs silhouette score
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(list(rango_k), silhouettes, 'rs-', linewidth=2, markersize=8)
ax.set_xlabel('Número de clusters (K)')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score por K')
ax.set_xticks(list(rango_k))
plt.tight_layout()
plt.show()

# 4. ¿Cuál K maximiza el silhouette?
mejor_k = list(rango_k)[np.argmax(silhouettes)]
mejor_sil = max(silhouettes)
print(f'\n📌 El K que maximiza el silhouette score es K={mejor_k} ({mejor_sil:.4f})')

# 5. ¿Coincide con el codo?
print(f'   Usaremos K=3 para el análisis, que es consistente con ambos métodos.')

---

## Ejercicio 4: Ajustar K-Means y asignar etiquetas

In [ ]:
# ============================================================
# SOLUCIÓN Ejercicio 4: Ajustar K-Means y asignar etiquetas
# ============================================================

# 1. Ajustar K-Means con K=3
K = 3
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
kmeans.fit(X_scaled)

# 2. Agregar la columna cluster al DataFrame original
df_cluster['cluster'] = kmeans.labels_

# 3. Cuántos docentes hay en cada cluster
print('📊 Distribución de docentes por cluster:')
print(df_cluster['cluster'].value_counts().sort_index())

# 4. Visualizar con PCA (reducción a 2 dimensiones)
#    PCA permite proyectar los 7 features en 2D para visualización
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1],
                     c=kmeans.labels_, cmap='Set2',
                     alpha=0.7, edgecolors='k', linewidth=0.5)

# Agregar centroides proyectados
centroides_pca = pca.transform(kmeans.cluster_centers_)
ax.scatter(centroides_pca[:, 0], centroides_pca[:, 1],
           c='red', marker='X', s=200, edgecolors='k', linewidth=2,
           label='Centroides')

ax.set_xlabel(f'Componente Principal 1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)')
ax.set_ylabel(f'Componente Principal 2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)')
ax.set_title('Clusters de Perfiles Docentes (PCA 2D)')
ax.legend()
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.show()

print(f'\n📌 Varianza explicada por las 2 componentes: '
      f'{pca.explained_variance_ratio_.sum()*100:.1f}%')

---

## Ejercicio 5: Tabla de perfiles

In [ ]:
# ============================================================
# SOLUCIÓN Ejercicio 5: Tabla de perfiles
# ============================================================

# 1. Calcular la media de cada variable por cluster
perfiles = df_cluster.groupby('cluster').mean()

# 2. Agregar el número de docentes por cluster
perfiles['n_docentes'] = df_cluster.groupby('cluster').size()

# 3. Redondear a 2 decimales
perfiles = perfiles.round(2)

# 4. Mostrar la tabla transpuesta para facilitar la lectura
print('📊 Tabla de perfiles por cluster (promedios):')
display(perfiles.T)

# 5. Identificar clusters clave
cluster_baja_satisfaccion = perfiles['satisfaccion_general'].idxmin()
cluster_mas_publicaciones = perfiles['publicaciones'].idxmax()

print(f'\n📌 Hallazgos:')
print(f'  - Cluster con satisfacción más baja: Cluster {cluster_baja_satisfaccion}')
print(f'    (satisfacción general promedio: {perfiles.loc[cluster_baja_satisfaccion, "satisfaccion_general"]:.2f})')
print(f'  - Cluster con más publicaciones: Cluster {cluster_mas_publicaciones}')
print(f'    (publicaciones promedio: {perfiles.loc[cluster_mas_publicaciones, "publicaciones"]:.2f})')

---

## Ejercicio 6: Nombrar clusters y escribir recomendación

In [ ]:
# ============================================================
# SOLUCIÓN Ejercicio 6: Nombrar clusters y recomendación
# ============================================================

# 1. Nombrar los clusters basándose en la tabla de perfiles
#    (los nombres dependen de los datos; estos son ejemplos razonables)
print('📊 Perfiles para referencia:')
display(perfiles.T)

# Asignar nombres basados en las características predominantes
# Nota: los nombres exactos dependerán de los valores que arroje el dataset
nombres_clusters = {}
for c in range(K):
    perfil = perfiles.loc[c]
    sat = perfil['satisfaccion_general']
    pub = perfil['publicaciones']
    carga = perfil['carga_horaria']
    ant = perfil['antiguedad']
    
    # Lógica de nombrado basada en características
    if sat >= perfiles['satisfaccion_general'].max():
        nombres_clusters[c] = 'Docentes satisfechos y estables'
    elif pub >= perfiles['publicaciones'].max():
        nombres_clusters[c] = 'Docentes productivos en investigación'
    elif sat <= perfiles['satisfaccion_general'].min():
        nombres_clusters[c] = 'Docentes en riesgo (baja satisfacción)'
    else:
        nombres_clusters[c] = f'Docentes perfil intermedio'

print('\n🏷️ Nombres asignados a los clusters:')
for c, nombre in nombres_clusters.items():
    n = perfiles.loc[c, 'n_docentes']
    print(f'  Cluster {c}: {nombre} ({int(n)} docentes)')

# 2. Identificar el cluster de mayor riesgo
cluster_riesgo = perfiles['satisfaccion_general'].idxmin()
print(f'\n⚠️ Cluster de mayor riesgo: Cluster {cluster_riesgo}')
print(f'   ({nombres_clusters[cluster_riesgo]})')

# 3. Recomendación para el Decanato
p = perfiles.loc[cluster_riesgo]
print(f'\n📋 Recomendación para el Decanato de FACES:')
print(f'''
El Cluster {cluster_riesgo} ({nombres_clusters[cluster_riesgo]}) agrupa a 
{int(p["n_docentes"])} docentes con una satisfacción general promedio de {p["satisfaccion_general"]:.1f}/5 
y satisfacción salarial de {p["satisfaccion_salarial"]:.1f}/5. Este grupo tiene una carga horaria 
promedio de {p["carga_horaria"]:.0f} horas. Este perfil representa un riesgo para la institución 
porque docentes insatisfechos tienden a reducir su productividad, su compromiso con la docencia, 
y eventualmente buscan oportunidades fuera de la universidad. Se recomienda implementar un 
programa de acompañamiento que incluya revisión de cargas horarias, oportunidades de 
desarrollo profesional, y espacios de diálogo para atender sus necesidades específicas.
''')

---

## 📋 Resumen

En este laboratorio practicaste:

| Paso | Qué hiciste |
|------|-------------|
| 1. Preparación | Seleccionaste, limpiaste y estandarizaste datos |
| 2. Método del codo | Identificaste el K óptimo visualmente |
| 3. Silhouette | Confirmaste el K con una métrica cuantitativa |
| 4. K-Means | Ajustaste el modelo y visualizaste clusters |
| 5. Perfilamiento | Construiste una tabla de perfiles por cluster |
| 6. Comunicación | Nombraste los clusters y escribiste una recomendación |

## 📚 Referencias

1. Scikit-learn developers. (2024). *KMeans*. https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html

2. Jain, A. K. (2010). *Data clustering: 50 years beyond K-means*. Pattern Recognition Letters, 31(8), 651-666.

---

*Notebook desarrollado para el programa de Formación Docente en Ciencia de Datos y Business Intelligence — FACES, Universidad Catolica Andres Bello (UCAB).*